In [ ]:
import os
print(os.listdir())

['.config', 'medicine.csv', 'sample_data']


In [ ]:
import pandas as pd
df = pd.read_csv("medicine.csv", encoding='latin1')
print(df.head())

   product_id                brand_name                          manufacturer  \
0           1  Augmentin 625 Duo Tablet  Glaxo SmithKline Pharmaceuticals Ltd   
1           2       Azithral 500 Tablet           Alembic Pharmaceuticals Ltd   
2           3          Ascoril LS Syrup          Glenmark Pharmaceuticals Ltd   
3           4      Allegra 120mg Tablet                      Sanofi India Ltd   
4           5            Avil 25 Tablet                      Sanofi India Ltd   

   price_inr  is_discontinued dosage_form  pack_size pack_unit  \
0     223.42            False      tablet       10.0     strip   
1     132.36            False      tablet        5.0     strip   
2     118.00            False       syrup      100.0    bottle   
3     218.81            False      tablet       10.0     strip   
4      10.96            False      tablet       15.0     strip   

   num_active_ingredients primary_ingredient primary_strength  \
0                       2        Amoxycillin       

In [ ]:
df['chemical'] = df['primary_ingredient'].str.strip().str.lower()
print(df[['primary_ingredient', 'chemical']].head())

  primary_ingredient      chemical
0        Amoxycillin   amoxycillin
1       Azithromycin  azithromycin
2           Ambroxol      ambroxol
3       Fexofenadine  fexofenadine
4        Pheniramine   pheniramine


In [ ]:
chem_counts = df['chemical'].value_counts().reset_index()
chem_counts.columns = ['chemical', 'demand_count']
print(chem_counts.head())

               chemical  demand_count
0           aceclofenac          4317
1              cefixime          3889
2           amoxycillin          3367
3          azithromycin          3075
4  cefpodoxime proxetil          2660


In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

le = LabelEncoder()
chem_counts['chemical_encoded'] = le.fit_transform(chem_counts['chemical'])

X = chem_counts[['chemical_encoded']]
y = chem_counts['demand_count']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestRegressor()
model.fit(X_train_scaled, y_train)

print("Model trained ✅")

Model trained ✅


In [ ]:
def predict_demand(name):
    name = name.lower()
    if name not in le.classes_:
        return "Not found"

    encoded = le.transform([name]).reshape(-1, 1)
    scaled = scaler.transform(encoded)

    return model.predict(scaled)[0]

print(predict_demand("amoxycillin"))

2245.34


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [ ]:
def predict_demand(name):
    name = name.lower()
    if name not in le.classes_:
        return "Not found"

    encoded = le.transform([name])

    import pandas as pd
    encoded_df = pd.DataFrame(encoded, columns=['chemical_encoded'])

    scaled = scaler.transform(encoded_df)

    return model.predict(scaled)[0]

In [ ]:
print(predict_demand("amoxycillin"))
print(predict_demand("azithromycin"))

2245.34
1950.2


In [ ]:
def demand_level(value):
    if value > 50:
        return "High"
    elif value > 20:
        return "Medium"
    else:
        return "Low"

value = predict_demand("amoxycillin")
print(value, demand_level(value))

2245.34 High


In [ ]:
import joblib
import json

joblib.dump(model, "model.pkl")
joblib.dump(le, "label_encoder.pkl")
joblib.dump(scaler, "scaler.pkl")

chem_counts.head(10).to_json("top_materials.json", orient="records")

In [ ]:
import zipfile

with zipfile.ZipFile('pharma_project.zip', 'w') as z:
    z.write('model.pkl')
    z.write('label_encoder.pkl')
    z.write('scaler.pkl')
    z.write('top_materials.json')

from google.colab import files
files.download("pharma_project.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
print(os.listdir())

['.config', 'medicine.csv', 'top_materials.json', 'label_encoder.pkl', 'scaler.pkl', 'pharma_project.zip', 'model.pkl', 'sample_data']
